In [ ]:
import requests
import pandas as pd

# 1. Requisição e extração da lista pura
api_response = requests.get("https://api.jolpi.ca/ergast/f1.json")
races_list = api_response.json()["MRData"]["RaceTable"]["Races"]

# 2. O Golpe de Mestre: Normalizando tudo de uma vez só
df_races = pd.json_normalize(races_list)

# 3. Limpeza do ruído (Cortando o peso inútil)
colunas_para_dropar = [
    "url", 
    "Circuit.url", 
    "Circuit.Location.lat", 
    "Circuit.Location.long",
]
df_races_clean = df_races.drop(columns=colunas_para_dropar)

# 4. Organizando a casa (Renomeando para o Merge perfeito)
df_f1_final = df_races_clean.rename(columns={
    "Circuit.circuitId": "circuitId",
    "Circuit.circuitName": "circuitName",
    "Circuit.Location.locality": "locality",
    "Circuit.Location.country": "country"
})

df_f1_final

In [ ]:
import requests
import json
import pandas as pd

dados_f1 = requests.get("https://api.jolpi.ca/ergast/f1/results.json")
dados_convertidos_json = dados_f1.json()


geral_statistics = dados_convertidos_json["MRData"]["RaceTable"]["Races"]

dataframe_jason = pd.json_normalize(geral_statistics, record_path="Results", meta=["round", "season"])

df_f1 = dataframe_jason.drop(columns=["Driver.url", "Constructor.url", "positionText"])

df_f1

In [13]:
df_f1_merged = df_f1.merge(df_f1_final, on=["round", "season"])

df_f1_merged

df_f1_merged

df_f1_merged["number"] = pd.to_numeric(df_f1_merged["number"], errors='coerce')
colunas = ["number", "position", "points", "grid", "laps", "Time.millis", "round", "season", "Time.time"]

df_f1_merged[colunas] = df_f1_merged[colunas].apply(pd.to_numeric, errors='coerce')
df_f1_merged["date"] = pd.to_datetime(df_f1_merged["date"])
df_f1_merged["Driver.dateOfBirth"] = pd.to_datetime(df_f1_merged["Driver.dateOfBirth"])


print(df_f1_merged.dtypes)

number                                int64
position                              int64
points                                int64
grid                                  int64
laps                                  int64
status                                  str
Driver.driverId                         str
Driver.givenName                        str
Driver.familyName                       str
Driver.dateOfBirth           datetime64[us]
Driver.nationality                      str
Constructor.constructorId               str
Constructor.name                        str
Constructor.nationality                 str
Time.millis                         float64
Time.time                           float64
round                                 int64
season                                int64
raceName                                str
date                         datetime64[us]
circuitId                               str
circuitName                             str
locality                        